# 9장. 합성곱 신경망

이미지를 숫자 784개로 펴서 넣는 신경망과, 이웃한 픽셀을 함께 보는 합성곱 신경망을 비교한다.

**준비:** Fashion-MNIST의 학습용 파일 두 개(`train-images-idx3-ubyte.gz`, `train-labels-idx1-ubyte.gz`)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 받는 곳은 https://github.com/zalandoresearch/fashion-mnist 의 `data/fashion` 폴더다. 시험용 파일 두 개는 16장에서 쓴다. 끝까지 실행하는 데 20분쯤 걸린다.

## 9.1 이미지를 숫자 784개로 펴서 분류하기

In [ ]:
import copy, gzip, time
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

torch.set_num_threads(2)

def read_idx(path, offset):
    """idx 형식의 파일을 읽는다. 앞의 offset바이트는 머리말이다."""
    with gzip.open(path) as f:
        return np.frombuffer(f.read(), np.uint8, offset=offset)

images = read_idx("train-images-idx3-ubyte.gz", 16).reshape(-1, 28, 28)
labels = read_idx("train-labels-idx1-ubyte.gz", 8).astype(np.int64)
names = ["티셔츠", "바지", "스웨터", "드레스", "코트",
         "샌들", "셔츠", "운동화", "가방", "앵클부츠"]
print(images.shape, images.dtype, images.min(), images.max())
print(np.bincount(labels))
print(images[0, 12, 8:16])

In [ ]:
order = np.random.default_rng(0).permutation(60000)
train_rows, rest_rows, valid_rows = order[:6000], order[6000:50000], order[50000:]

def tensors(rows, mean=None, std=None):
    """고른 이미지를 (장수, 1, 28, 28) 모양의 실수 텐서로 바꾸고 표준화한다."""
    x = torch.tensor(images[rows] / 255, dtype=torch.float32)[:, None]
    if mean is None:
        mean, std = x.mean().item(), x.std().item()
    return (x - mean) / std, torch.tensor(labels[rows]), mean, std

X_tr, y_tr, pixel_mean, pixel_std = tensors(train_rows)
X_va, y_va, _, _ = tensors(valid_rows, pixel_mean, pixel_std)
blank = (0 - pixel_mean) / pixel_std               # 검은 바탕(0)이 표준화된 값
print(X_tr.shape, X_va.shape, len(rest_rows))
print(np.bincount(labels[train_rows]))
print(round(pixel_mean, 4), round(pixel_std, 4), round(blank, 4))

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.0]])            # 이미지 한 장, 클래스 3개의 점수
prob = torch.softmax(scores, dim=1)
print(prob, prob.sum().item())
answer = torch.tensor([0])                          # 정답이 0번 클래스일 때
print(-torch.log(prob[0, 0]).item(), F.cross_entropy(scores, answer).item())
print(F.cross_entropy(scores, torch.tensor([2])).item())

In [ ]:
def accuracy(net, x, y):
    """정확도(%). 1,000장씩 나눠 예측한다."""
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(x[k:k + 1000]).argmax(dim=1)
                           for k in range(0, len(x), 1000)])
    return (guess == y).float().mean().item() * 100

def fit(net, data=None, change=None, epochs=10, seed=0, lr=0.001):
    """Adam, 배치 크기 64로 학습시킨다.
    data: (학습 이미지, 학습 정답, 검증 이미지, 검증 정답). 없으면 6,000장과 10,000장.
    change: 배치의 이미지를 바꾸는 함수(자료 늘리기). 9.4절에서 쓴다."""
    x, y, x_va, y_valid = data or (X_tr, y_tr, X_va, y_va)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    shuffler = torch.Generator().manual_seed(seed)
    log = []
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(x), generator=shuffler)
        for start in range(0, len(x), 64):
            rows = mixed[start:start + 64]
            batch = x[rows] if change is None else change(x[rows], shuffler)
            optimizer.zero_grad()
            F.cross_entropy(net(batch), y[rows]).backward()
            optimizer.step()
        log.append((accuracy(net, x, y), accuracy(net, x_va, y_valid)))
    return np.array(log)

def trial(make_net, **options):
    """시드 0~4로 학습시킨다. 기록(5, 에폭, 2)과 학습된 모델 5개를 돌려준다."""
    logs, nets = [], []
    for seed in range(5):
        torch.manual_seed(seed)
        net = make_net()
        logs.append(fit(net, seed=seed, **options))
        nets.append(net)
    return np.array(logs), nets

def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def show(name, logs):
    last = logs[:, -1]
    span = f"({last[:, 1].min():.2f}~{last[:, 1].max():.2f})"
    middle = np.median(last, axis=0)
    print(pad(name, 30) + f"{middle[0]:>5.1f}   {middle[1]:.2f} {span}")

def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                         nn.Linear(128, 10))

count = lambda net: sum(param.numel() for param in net.parameters())
print(count(make_mlp()))
lap = time.perf_counter()
logs, nets = {}, {}
logs["펴서 넣는 신경망"], nets["펴서 넣는 신경망"] = trial(make_mlp)
print(pad("검증 10,000장의 정확도(%)", 30) + " 학습   검증 (범위)")
show("펴서 넣는 신경망", logs["펴서 넣는 신경망"])
print(f"{time.perf_counter() - lap:.0f}초")

In [ ]:
scramble = torch.randperm(784, generator=torch.Generator().manual_seed(0))  # 섞는 순서

def scrambled(x):
    """모든 이미지의 픽셀을 같은 순서로 섞는다."""
    return x.reshape(len(x), 784)[:, scramble].reshape(len(x), 1, 28, 28)

mixed_data = (scrambled(X_tr), y_tr, scrambled(X_va), y_va)
logs["펴서 넣는 신경망, 픽셀 섞음"], _ = trial(make_mlp, data=mixed_data)
show("펴서 넣는 신경망", logs["펴서 넣는 신경망"])
show("펴서 넣는 신경망, 픽셀 섞음", logs["펴서 넣는 신경망, 픽셀 섞음"])

## 9.2 합성곱을 넘파이로 만들기

In [ ]:
def conv2d(image, kernel):
    """이미지 위에서 커널을 한 칸씩 옮기며 겹친 값끼리 곱해 더한다."""
    k = len(kernel)
    rows, cols = image.shape[0] - k + 1, image.shape[1] - k + 1
    out = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            out[r, c] = (image[r:r + k, c:c + k] * kernel).sum()
    return out

small = np.array([[0, 0, 1, 1, 1],
                  [0, 0, 1, 1, 1],
                  [0, 0, 1, 1, 1],
                  [0, 0, 1, 1, 1],
                  [0, 0, 1, 1, 1]], dtype=np.float64)
edge = np.array([[-1, 0, 1],
                 [-1, 0, 1],
                 [-1, 0, 1]], dtype=np.float64)  # 왼쪽 값이 작고 오른쪽이 크면 큰 값
print(conv2d(small, edge))

In [ ]:
shoe = images[np.flatnonzero(labels == 7)[0]] / 255    # 첫 번째 운동화
by_hand = conv2d(shoe, edge)
by_torch = F.conv2d(torch.tensor(shoe)[None, None], torch.tensor(edge)[None, None])
print(by_hand.shape, float(np.abs(by_hand - by_torch[0, 0].numpy()).max()))
print(round(by_hand.min(), 2), round(by_hand.max(), 2))

moved = np.zeros_like(shoe)
moved[:, 2:] = shoe[:, :-2]                            # 오른쪽으로 2칸 옮긴 이미지
print(float(np.abs(conv2d(moved, edge)[:, 2:] - by_hand[:, :-2]).max()))

In [ ]:
def conv_layer(x, weight, bias):
    """x: (입력 채널, 높이, 너비). weight: (출력 채널, 입력 채널, 3, 3). 패딩 1."""
    padded = np.pad(x, ((0, 0), (1, 1), (1, 1)))
    out = np.zeros((len(weight), x.shape[1], x.shape[2]))
    for o in range(len(weight)):                       # 출력 채널마다
        for i in range(x.shape[0]):                    # 입력 채널마다 합성곱해서 더한다
            out[o] += conv2d(padded[i], weight[o, i])
        out[o] += bias[o]
    return out

torch.manual_seed(0)
layer = nn.Conv2d(1, 16, kernel_size=3, padding=1)
one = X_tr[0].numpy().astype(np.float64)               # (1, 28, 28)
mine = conv_layer(one, layer.weight.detach().numpy().astype(np.float64),
                  layer.bias.detach().numpy().astype(np.float64))
print(layer.weight.shape, layer.bias.shape, count(layer), mine.shape)
print(float(np.abs(mine - layer(X_tr[:1])[0].detach().numpy()).max()))

## 9.3 풀링을 더해 합성곱 신경망 만들기

In [ ]:
def max_pool(x):
    """(채널, 높이, 너비)에서 2×2 칸마다 가장 큰 값만 남긴다."""
    c, h, w = x.shape
    return x.reshape(c, h // 2, 2, w // 2, 2).max(axis=(2, 4))

sample = np.array([[[1, 3, 2, 0],
                    [4, 2, 1, 1],
                    [0, 1, 5, 6],
                    [2, 2, 7, 3]]], dtype=np.float64)
print(max_pool(sample))
print(float(np.abs(max_pool(mine) - F.max_pool2d(torch.tensor(mine), 2).numpy()).max()))

In [ ]:
def make_cnn(drop=0.0, norm=False):
    """합성곱(16) - 풀링 - 합성곱(32) - 풀링 - 선형. drop, norm은 9.4절에서 쓴다."""
    layers = [nn.Conv2d(1, 16, kernel_size=3, padding=1)]
    layers += [nn.BatchNorm2d(16)] if norm else []
    layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, kernel_size=3, padding=1)]
    layers += [nn.BatchNorm2d(32)] if norm else []
    layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Flatten()]
    layers += [nn.Dropout(drop)] if drop > 0 else []
    return nn.Sequential(*layers, nn.Linear(32 * 7 * 7, 10))

torch.manual_seed(0)
net = make_cnn()
out = X_tr[:2]
for layer in net:
    out = layer(out)
    print(pad(type(layer).__name__, 10), tuple(out.shape), count(layer))
print(count(net))

In [ ]:
no_padding = nn.Sequential(nn.Conv2d(1, 16, 3), nn.ReLU(), nn.MaxPool2d(2),
                           nn.Conv2d(16, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
                           nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
try:
    no_padding(X_tr[:2])
except RuntimeError as error:
    print("RuntimeError:", error)
print(tuple(no_padding[:7](X_tr[:2]).shape))

In [ ]:
lap = time.perf_counter()
logs["합성곱 신경망"], nets["합성곱 신경망"] = trial(make_cnn)
took = time.perf_counter() - lap
logs["합성곱 신경망, 픽셀 섞음"], _ = trial(make_cnn, data=mixed_data)
print(pad("검증 10,000장의 정확도(%)", 30) + " 학습   검증 (범위)")
for name in ["펴서 넣는 신경망", "펴서 넣는 신경망, 픽셀 섞음",
             "합성곱 신경망", "합성곱 신경망, 픽셀 섞음"]:
    show(name, logs[name])
print(f"{took:.0f}초")

In [ ]:
print("학습률     학습   검증 (시드 0)")
for rate in [0.0001, 0.001, 0.01, 0.1]:
    torch.manual_seed(0)
    record = fit(make_cnn(), lr=rate)
    print(f"{rate:<8}  {record[-1, 0]:>5.1f}   {record[-1, 1]:.2f}")

In [ ]:
def slide(x, cells):
    """모든 이미지를 오른쪽으로 cells칸 옮긴다. 빈 곳은 바탕 값으로 채운다."""
    if cells == 0:
        return x
    out = torch.full_like(x, blank)
    out[..., cells:] = x[..., :-cells]
    return out

slid = {}
print("오른쪽으로 옮긴 칸  " + "".join(f"{cells:>7}" for cells in range(5)))
for name in ["펴서 넣는 신경망", "합성곱 신경망"]:
    slid[name] = np.array([[accuracy(net, slide(X_va, cells), y_va)
                            for cells in range(5)] for net in nets[name]])
    middle = np.median(slid[name], axis=0)
    print(pad(name, 20) + "".join(f"{value:>7.2f}" for value in middle))

## 9.4 드롭아웃, 자료 늘리기, 배치 정규화 다시 확인하기

In [ ]:
def nudge(x, shuffler):
    """이미지마다 위아래와 좌우로 0~2칸을 무작위로 옮긴다."""
    wide = F.pad(x, (2, 2, 2, 2), value=blank)         # 사방에 바탕 2칸을 덧댄다
    moves = torch.randint(0, 5, (len(x), 2), generator=shuffler).tolist()
    return torch.stack([wide[k, :, dy:dy + 28, dx:dx + 28]
                        for k, (dy, dx) in enumerate(moves)])

def upside_down(x, shuffler):
    """이미지의 절반을 무작위로 골라 위아래를 뒤집는다."""
    turn = torch.rand(len(x), generator=shuffler) < 0.5
    return torch.where(turn[:, None, None, None], x.flip(2), x)

lap = time.perf_counter()
plans = [("합성곱 + 드롭아웃 0.5", lambda: make_cnn(drop=0.5), None),
         ("합성곱 + 조금 옮기기", make_cnn, nudge),
         ("합성곱 + 위아래 뒤집기", make_cnn, upside_down),
         ("합성곱 + 배치 정규화", lambda: make_cnn(norm=True), None)]
for name, make_net, change in plans:
    logs[name], nets[name] = trial(make_net, change=change)
print(pad("검증 10,000장의 정확도(%)", 30) + " 학습   검증 (범위)")
settings = ["합성곱 신경망"] + [name for name, _, _ in plans]
for name in settings:
    show(name, logs[name])
print(f"{time.perf_counter() - lap:.0f}초")

In [ ]:
first = [np.median(logs[name][:, 0, 1]) for name in settings]
print(pad("1에폭의 검증 정확도", 30) + "".join(f"{value:>7.2f}" for value in first))
print(pad("학습 정확도 - 검증 정확도", 30) + " 5에폭   10에폭")
for name in settings:
    apart = np.median(logs[name][:, :, 0] - logs[name][:, :, 1], axis=0)
    print(pad(name, 30) + f"{apart[4]:>6.2f}  {apart[9]:>6.2f}")
name = "합성곱 + 조금 옮기기"
slid[name] = np.array([[accuracy(net, slide(X_va, cells), y_va)
                        for cells in range(5)] for net in nets[name]])
middle = np.median(slid[name], axis=0)
print(pad("조금 옮기기, 옮긴 칸 0~4", 30) +
      "".join(f"{value:>7.2f}" for value in middle))

In [ ]:
net = copy.deepcopy(nets["합성곱 + 배치 정규화"][0])
print(f"{accuracy(net, X_va, y_va):.2f}")         # 평가 모드
for size in [1000, 16, 2]:
    net = copy.deepcopy(nets["합성곱 + 배치 정규화"][0])
    net.train()                              # 학습 모드: 배치의 평균과 분산을 쓴다
    with torch.no_grad():
        guess = torch.cat([net(X_va[k:k + size]).argmax(dim=1)
                           for k in range(0, 10000, size)])
    print(size, f"{(guess == y_va).float().mean().item() * 100:.2f}")

by_class = torch.argsort(y_va, stable=True)       # 검증 자료를 정답 순서로 늘어놓는다
groups = y_va[by_class].reshape(100, 100)         # 100장씩 100개 배치의 정답
pure = int((groups.min(dim=1).values == groups.max(dim=1).values).sum())
print(pure)                                       # 한 클래스만 든 배치의 수
for mode in ["평가 모드", "학습 모드"]:
    net = copy.deepcopy(nets["합성곱 + 배치 정규화"][0])
    net.train(mode == "학습 모드")
    with torch.no_grad():
        guess = torch.cat([net(X_va[by_class[k:k + 100]]).argmax(dim=1)
                           for k in range(0, 10000, 100)])
    print(mode, f"{(guess == y_va[by_class]).float().mean().item() * 100:.2f}")

In [ ]:
lap = time.perf_counter()
longer = {}
for name, change in [("합성곱 신경망", None), ("합성곱 + 조금 옮기기", nudge),
                     ("합성곱 + 위아래 뒤집기", upside_down)]:
    torch.manual_seed(0)
    longer[name] = fit(make_cnn(), change=change, epochs=30)
print(pad("시드 0, 에폭", 24) + "     10             20             30")
for name, record in longer.items():
    print(pad(name, 24) + "".join(f"  {record[e, 0]:>5.1f} {record[e, 1]:>6.2f}"
                                  for e in [9, 19, 29]))
    print(pad("  검증이 가장 높은 에폭", 24) +
          f"  {record[:, 1].argmax() + 1}에폭 {record[:, 1].max():.2f}")
print(f"{time.perf_counter() - lap:.0f}초")

## 9.5 클래스별 결과와 자료를 모두 썼을 때

In [ ]:
def table_of(net):
    """행은 정답, 열은 예측인 표(혼동 행렬)."""
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(X_va[k:k + 1000]).argmax(dim=1)
                           for k in range(0, 10000, 1000)])
    return np.bincount(y_va.numpy() * 10 + guess.numpy(), minlength=100).reshape(10, 10)

def average_table(name):
    """시드 5개의 표를 평균한다."""
    return np.mean([table_of(net) for net in nets[name]], axis=0)

confusion, plain = average_table("합성곱 신경망"), average_table("펴서 넣는 신경망")
total = confusion.sum(axis=1)
print(total.astype(int).tolist(), int(total.sum()))
print("클래스      합성곱   펴서 넣기")
for k in range(10):
    print(pad(names[k], 10) + f"{confusion[k, k] / total[k] * 100:>6.1f}   "
          f"{plain[k, k] / total[k] * 100:>6.1f}")
wrong = confusion - np.diag(np.diag(confusion))
for flat in np.argsort(-wrong.ravel())[:4]:
    truth, said = divmod(int(flat), 10)
    print(f"{names[truth]}를 {names[said]}로: {wrong[truth, said]:.0f}장")

In [ ]:
full_rows = order[:50000]
X_full, y_full, full_mean, full_std = tensors(full_rows)
X_va_full = tensors(valid_rows, full_mean, full_std)[0]
full_data = (X_full, y_full, X_va_full, y_va)
full = {}
for name, make_net in [("펴서 넣는 신경망", make_mlp), ("합성곱 신경망", make_cnn)]:
    lap = time.perf_counter()
    torch.manual_seed(0)
    net = make_net()
    full[name] = fit(net, data=full_data)
    print(pad(name, 20) + f"{full[name][-1, 0]:.1f}   {full[name][-1, 1]:.2f}   "
          f"{time.perf_counter() - lap:.0f}초")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

flat_tr, flat_va = X_tr.reshape(6000, 784).numpy(), X_va.reshape(10000, 784).numpy()
models = [("로지스틱 회귀", LogisticRegression(max_iter=1000)),
          ("그래디언트 부스팅", HistGradientBoostingClassifier(random_state=0))]
for name, model in models:
    lap = time.perf_counter()
    model.fit(flat_tr, y_tr.numpy())
    on_train = model.score(flat_tr, y_tr.numpy()) * 100
    on_valid = model.score(flat_va, y_va.numpy()) * 100
    print(pad(name, 20) + f"{on_train:.1f}   {on_valid:.2f}   "
          f"{time.perf_counter() - lap:.0f}초")

In [ ]:
lap = time.perf_counter()
model = HistGradientBoostingClassifier(random_state=0)
model.fit(X_full.reshape(50000, 784).numpy(), y_full.numpy())
on_valid = model.score(X_va_full.reshape(10000, 784).numpy(), y_va.numpy()) * 100
took = time.perf_counter() - lap
print(pad("그래디언트 부스팅", 20) + f"{on_valid:.2f}   {took:.0f}초")

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, torch.__version__, sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 9-1
from matplotlib.patches import Rectangle

tops, shoes = [0, 2, 4, 6], [5, 7, 9]
tone = lambda k: BRICK if k in tops else TEAL if k in shoes else NAVY
fig = plt.figure(figsize=(7, 7.2))
upper = fig.add_gridspec(4, 10, left=0.02, right=0.98, top=0.96, bottom=0.53, wspace=0.08, hspace=0.08)
for k in range(10):
    found = np.flatnonzero(labels[train_rows] == k)[:4]
    for r in range(4):
        ax = fig.add_subplot(upper[r, k])
        ax.imshow(images[train_rows[found[r]]], cmap="gray_r", vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        for side in ax.spines.values():
            side.set_color(tone(k)); side.set_linewidth(0.9)
        if r == 0:
            ax.set_title(f"{k} {names[k]}", fontsize=9, color=tone(k), weight="bold")
fig.text(0.02, 0.500, "■ 윤곽이 비슷한 윗옷 네 클래스", fontsize=9, color=BRICK)
fig.text(0.40, 0.500, "■ 신발 세 클래스", fontsize=9, color=TEAL)
fig.text(0.63, 0.500, "■ 바지, 드레스, 가방", fontsize=9, color=NAVY)
fig.text(0.02, 0.455, "이미지 한 장은 숫자 784개다: 첫 번째 운동화와 그 일부의 값", fontsize=9, color=NAVY,
         weight="bold")
big = fig.add_axes([0.085, 0.065, 0.345, 0.345])
first_shoe = images[np.flatnonzero(labels == 7)[0]]
big.imshow(first_shoe, cmap="gray_r", vmin=0, vmax=255)
big.set_xticks(np.arange(-0.5, 28), minor=True); big.set_yticks(np.arange(-0.5, 28), minor=True)
big.grid(which="minor", color="#d5dae1", lw=0.3); big.tick_params(which="both", length=0)
big.set_xticks([0, 9, 18, 27]); big.set_yticks([0, 9, 18, 27])
big.set_xticklabels([1, 10, 19, 28], fontsize=9); big.set_yticklabels([1, 10, 19, 28], fontsize=9)
big.set_xlabel("열 (28칸)", fontsize=9); big.set_ylabel("행 (28칸)", fontsize=9)
top_row, left_col = 9, 9                                # 숫자로 보여 줄 8×8 칸의 왼쪽 위
big.add_patch(Rectangle((left_col - 0.5, top_row - 0.5), 8, 8, fc="none", ec=BRICK, lw=1.6))
small_ax = fig.add_axes([0.585, 0.065, 0.345, 0.345])
part = first_shoe[top_row:top_row + 8, left_col:left_col + 8]
small_ax.imshow(part, cmap="gray_r", vmin=0, vmax=255)
for r in range(8):
    for c in range(8):
        small_ax.text(c, r, str(part[r, c]), fontsize=9, ha="center", va="center",
                      color="white" if part[r, c] > 130 else "#3a4350")
small_ax.set_xticks(range(8)); small_ax.set_yticks(range(8))
small_ax.set_xticklabels(range(left_col + 1, left_col + 9), fontsize=9)
small_ax.set_yticklabels(range(top_row + 1, top_row + 9), fontsize=9)
small_ax.tick_params(length=0)
for side in small_ax.spines.values():
    side.set_color(BRICK); side.set_linewidth(1.6)
small_ax.set_title("벽돌색 네모 안의 값 (0은 바탕, 255가 가장 진하다)", fontsize=9, color=BRICK)
small_ax.set_xlabel("열", fontsize=9); small_ax.set_ylabel("행", fontsize=9)
big.set_title("운동화 (28×28)", fontsize=9, color="#3a4350")
fig.savefig("fig_9_1.png", dpi=300)
plt.show()
print(int(part.min()), int(part.max()), int((part == 0).sum()))

In [ ]:
# 그림 9-3
fig, axes = plt.subplots(2, 4, figsize=(7, 4.5))
picks = [np.flatnonzero(labels[train_rows] == k)[0] for k in [7, 1, 8, 3]]
traced = [(14, 5, TEAL), (14, 14, BRICK), (8, 20, NAVY)]       # 따라가 볼 픽셀 세 개 (행, 열, 색)
landed = {int(old): new for new, old in enumerate(scramble.tolist())}   # 원래 번호 -> 섞인 뒤 번호
for col, row in enumerate(picks):
    before = X_tr[row:row + 1]
    for r, picture in enumerate([before, scrambled(before)]):
        axes[r, col].imshow(picture[0, 0], cmap="gray_r")
        axes[r, col].set_xticks([]); axes[r, col].set_yticks([])
        for side in axes[r, col].spines.values():
            side.set_color("#c9ced6"); side.set_linewidth(0.6)
        for y, x, shade in traced:
            if r == 1:
                y, x = divmod(landed[y * 28 + x], 28)
            axes[r, col].add_patch(Rectangle((x - 1, y - 1), 2, 2, fc="none", ec=shade, lw=1.5))
    axes[0, col].set_title(names[int(y_tr[row])], fontsize=9, color=NAVY, weight="bold")
axes[0, 0].set_ylabel("원래 이미지", fontsize=9, color=NAVY)
axes[1, 0].set_ylabel("픽셀을 섞은 이미지", fontsize=9, color=BRICK)
fig.tight_layout(pad=0.5, h_pad=2.6, rect=(0, 0.07, 1, 1))
fig.text(0.5, 0.535, "▼  784개의 순서를 모든 이미지에서 똑같이 섞는다. 색 네모 세 개는 같은 픽셀이 옮겨 간 곳이다.",
         fontsize=9, color="#3a4350", ha="center", va="center")
plain_acc = np.median(logs["펴서 넣는 신경망"][:, -1, 1])
mixed_acc = np.median(logs["펴서 넣는 신경망, 픽셀 섞음"][:, -1, 1])
fig.text(0.5, 0.03, f"펴서 넣는 신경망의 검증 정확도: 원래 이미지로 학습 {plain_acc:.2f}%,  섞은 이미지로 학습 {mixed_acc:.2f}%",
         fontsize=9, color=NAVY, ha="center", va="center", weight="bold")
fig.savefig("fig_9_3.png", dpi=300)
plt.show()
print([divmod(landed[y * 28 + x], 28) for y, x, _ in traced])

In [ ]:
# 그림 9-5
from matplotlib.colors import LinearSegmentedColormap

two_way = LinearSegmentedColormap.from_list("two_way", [BRICK, "white", NAVY])
flat_edge = conv2d(shoe, edge.T)                    # 커널의 행과 열을 바꿨다
fig = plt.figure(figsize=(7, 4.5))
panels = fig.add_gridspec(2, 3, height_ratios=[1, 3.1], left=0.02, right=0.98, top=0.93, bottom=0.14,
                          wspace=0.06, hspace=0.16)
note = fig.add_subplot(panels[0, 0]); note.axis("off")
note.text(0.5, 0.62, "커널 두 개", fontsize=9, color=NAVY, ha="center", weight="bold")
note.text(0.5, 0.30, "같은 운동화에\n커널만 바꿔 넣는다", fontsize=9, color="#3a4350", ha="center",
          va="center", linespacing=1.5)
for col, kernel, word in [(1, edge, "왼쪽 열 $-$1, 오른쪽 열 1"), (2, edge.T, "위 행 $-$1, 아래 행 1")]:
    ax = fig.add_subplot(panels[0, col])
    ax.imshow(kernel, cmap=two_way, vmin=-2.2, vmax=2.2)
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f"{kernel[r, c]:.0f}".replace("-", "$-$"), fontsize=9, ha="center", va="center",
                    color="#3a4350")
    ax.set_xticks(np.arange(-0.5, 3), minor=True); ax.set_yticks(np.arange(-0.5, 3), minor=True)
    ax.grid(which="minor", color="white", lw=1.2); ax.tick_params(which="both", length=0)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"커널: {word}", fontsize=9, color="#3a4350")
    for side in ax.spines.values():
        side.set_visible(False)
shown_in = fig.add_subplot(panels[1, 0])
shown_in.imshow(shoe, cmap="gray_r", vmin=0, vmax=1)
shown_in.set_title("입력 (28×28): 진할수록 큰 값", fontsize=9, color="#3a4350")
marks = []
for col, result, plus, minus in [(1, by_hand, "오른쪽 값이 더 크다", "오른쪽 값이 더 작다"),
                                 (2, flat_edge, "아래쪽 값이 더 크다", "아래쪽 값이 더 작다")]:
    ax = fig.add_subplot(panels[1, col])
    shown = ax.imshow(result, cmap=two_way, vmin=-3, vmax=3)
    ax.set_title("출력 (26×26)", fontsize=9, color="#3a4350")
    high = np.unravel_index(result.argmax(), result.shape)
    low = np.unravel_index(result.argmin(), result.shape)
    marks.append((tuple(int(v) for v in high), tuple(int(v) for v in low)))
    spots = [(12.4, 2.6), (0.5, 23.2)] if col == 1 else [(0.5, 3.6), (0.5, 23.2)]   # 글을 적을 곳
    ax.annotate(f"가장 큰 값 {result.max():.2f}\n{plus}", xy=(high[1], high[0]), xytext=spots[0], fontsize=9,
                color=NAVY, va="center", linespacing=1.35,
                arrowprops=dict(arrowstyle="->", color=NAVY, lw=0.8))
    ax.annotate(f"가장 작은 값 {result.min():.2f}\n{minus}".replace("-", "$-$"), xy=(low[1], low[0]),
                xytext=spots[1], fontsize=9, color=BRICK, va="center", linespacing=1.35,
                arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.8))
for ax in fig.axes[3:]:
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_color("#c9ced6"); side.set_linewidth(0.6)
bar = fig.colorbar(shown, cax=fig.add_axes([0.42, 0.075, 0.5, 0.025]), orientation="horizontal")
bar.ax.tick_params(labelsize=9); bar.outline.set_visible(False)
bar.set_ticks([-3, -2, -1, 0, 1, 2, 3]); bar.set_ticklabels(["$-$3", "$-$2", "$-$1", "0", "1", "2", "3"])
fig.text(0.40, 0.088, "출력 값 (벽돌색 음수, 흰색 0, 남색 양수)", fontsize=9, color="#3a4350", ha="right",
         va="center")
fig.savefig("fig_9_5.png", dpi=300)
plt.show()
print(round(float(flat_edge.min()), 2), round(float(flat_edge.max()), 2), marks)

In [ ]:
# 그림 9-7
fig = plt.figure(figsize=(7, 5.6))
ax = fig.add_axes([0.09, 0.30, 0.88, 0.67])
for name, col, lift in [("펴서 넣는 신경망", BRICK, -5.2), ("합성곱 신경망", NAVY, 3.4)]:
    middle = np.median(slid[name], axis=0)
    for row in slid[name]:
        ax.plot(range(5), row, color=col, lw=0.6, alpha=0.45)
    ax.plot(range(5), middle, color=col, lw=2.2, marker="o", ms=5, label=name)
    for cells, value in enumerate(middle):
        ax.text(cells + 0.05, value + lift, f"{value:.1f}", fontsize=9, color=col, weight="bold",
                ha="left" if lift > 0 else "right", va="center", bbox=BOX)
ax.axhline(10, color=GRAY, lw=0.9, ls="--")
ax.text(-0.25, 12.4, "아무 클래스나 고를 때 10%", fontsize=9, color="#5b6470")
ax.set_xticks(range(5)); ax.set_xlim(-0.3, 4.3); ax.set_ylim(0, 100)
ax.set_ylabel("검증 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right", title="굵은 선: 시드 5개의 중앙값, 가는 선: 시드마다",
          title_fontsize=9)
frame(ax)
sample = int(np.flatnonzero(y_va.numpy() == 7)[0])                 # 검증 자료의 첫 운동화
fig.canvas.draw()
for cells in range(5):
    spot = fig.transFigure.inverted().transform(ax.transData.transform((cells, 0)))[0]
    thumb = fig.add_axes([spot - 0.062, 0.075, 0.124, 0.155])
    thumb.imshow(slide(X_va[sample:sample + 1], cells)[0, 0], cmap="gray_r")
    thumb.axvline(13.5, color=BRICK, lw=0.6, ls=":")
    thumb.set_xticks([]); thumb.set_yticks([])
    for side in thumb.spines.values():
        side.set_color("#9aa1ab"); side.set_linewidth(0.7)
    thumb.set_xlabel("그대로" if cells == 0 else f"{cells}칸", fontsize=9)
fig.text(0.53, 0.012, "검증 이미지를 오른쪽으로 옮긴 칸 수 (점선은 이미지의 가운데)", fontsize=9, ha="center")
fig.savefig("fig_9_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 9-8
shown_classes = [7, 9, 0, 6, 1, 8]
sample_rows = [int(np.flatnonzero(labels[train_rows] == k)[1]) for k in shown_classes]
before = X_tr[sample_rows]
moved = nudge(before, torch.Generator().manual_seed(1))
moves = torch.randint(0, 5, (6, 2), generator=torch.Generator().manual_seed(1)).tolist()

def in_words(dy, dx):
    # nudge가 고른 (dy, dx)를 옮긴 방향으로 적는다. 2에서 뺀 값이 아래쪽과 오른쪽으로 옮긴 칸 수다.
    down, right = 2 - dy, 2 - dx
    across = "가로 0칸" if right == 0 else f"{'오른쪽' if right > 0 else '왼쪽'} {abs(right)}칸"
    upright = "세로 0칸" if down == 0 else f"{'아래' if down > 0 else '위'} {abs(down)}칸"
    return across + ", " + upright

fig, axes = plt.subplots(3, 6, figsize=(7, 4.9))
fig.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.04, wspace=0.08, hspace=0.46)
kinds = [("원래 이미지", before, "#3a4350"),
         ("조금 옮긴 이미지: 무엇인지 그대로 알 수 있다. 검증 자료에도 있을 법하다", moved, TEAL),
         ("위아래를 뒤집은 이미지: 검증 자료에는 이런 이미지가 없다", before.flip(2), BRICK)]
for r, (word, pictures, col) in enumerate(kinds):
    for c in range(6):
        ax = axes[r, c]
        ax.imshow(pictures[c, 0], cmap="gray_r", vmin=blank, vmax=(1 - pixel_mean) / pixel_std)
        ax.axvline(13.5, color=col, lw=0.5, ls=":"); ax.axhline(13.5, color=col, lw=0.5, ls=":")
        ax.set_xticks([]); ax.set_yticks([])
        for side in ax.spines.values():
            side.set_color(col); side.set_linewidth(0.9)
        if r == 0:
            ax.set_xlabel(names[shown_classes[c]], fontsize=9, color=col)
        if r == 1:
            ax.set_xlabel(in_words(*moves[c]).replace(", ", "\n"), fontsize=9, color=col)
    axes[r, 0].set_title(word, fontsize=9, color=col, loc="left", weight="bold")
fig.savefig("fig_9_8.png", dpi=300)
plt.show()
print([in_words(*move) for move in moves])

In [ ]:
# 그림 9-9
fig, axes = plt.subplots(2, 1, figsize=(7, 7.6), sharex=True)
styles = {"합성곱 신경망": ("#3a4350", "-", 2.6), "합성곱 + 드롭아웃 0.5": (GRAY, "-", 1.8),
          "합성곱 + 조금 옮기기": (TEAL, "-", 1.8), "합성곱 + 위아래 뒤집기": (BRICK, "-", 1.8),
          "합성곱 + 배치 정규화": (NAVY, "--", 1.8)}
panels = [(axes[0], 0, "학습 자료 6,000장의 정확도", (72, 100)),
          (axes[1], 1, "검증 자료 10,000장의 정확도", (72, 90))]
for ax, column, word, limits in panels:
    curves = {name: np.median(logs[name][:, :, column], axis=0) for name in styles}
    for name in sorted(curves, key=lambda name: -curves[name][-1]):      # 10에폭의 값이 큰 순서
        col, dash, width = styles[name]
        ax.plot(range(1, 11), curves[name], color=col, ls=dash, lw=width, marker="o", ms=3.2,
                label=f"{name}  {curves[name][-1]:.1f}%")
    ax.set_ylabel("정확도 (%)", fontsize=9)
    ax.set_xticks(range(1, 11)); ax.set_ylim(*limits)
    ax.legend(frameon=False, fontsize=9, loc="lower right", title="10에폭의 값이 큰 순서", title_fontsize=9)
    frame(ax, title=word)
axes[0].tick_params(labelbottom=True)
axes[1].set_xlabel("에폭", fontsize=9)
fig.tight_layout(h_pad=1.6)
fig.savefig("fig_9_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 9-10
share = confusion / total[:, None] * 100
shown_order = [0, 2, 4, 6, 3, 1, 8, 5, 7, 9]           # 윗옷 넷, 드레스·바지·가방, 신발 셋의 순서로 놓는다
moved_share = share[np.ix_(shown_order, shown_order)]
fig, ax = plt.subplots(figsize=(7, 6.6))
ax.imshow(moved_share, cmap=LinearSegmentedColormap.from_list("ink", ["white", NAVY]), vmin=0, vmax=100)
for r in range(10):
    for c in range(10):
        if moved_share[r, c] < 0.5:
            continue
        loud = r != c and round(moved_share[r, c]) >= 5
        col = "white" if moved_share[r, c] > 55 else (BRICK if loud else "#3a4350")
        ax.text(c, r, f"{moved_share[r, c]:.0f}", fontsize=9, color=col, ha="center", va="center",
                weight="bold" if loud or r == c else None)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels([names[k] for k in shown_order], fontsize=9, rotation=45, ha="right")
ax.set_yticklabels([names[k] for k in shown_order], fontsize=9)
for tick_set in (ax.get_xticklabels(), ax.get_yticklabels()):
    for k, tick in zip(shown_order, tick_set):
        tick.set_color(tone(k))
ax.set_xlabel("모델이 고른 클래스", fontsize=9); ax.set_ylabel("정답 클래스", fontsize=9)
ax.set_xticks(np.arange(-0.5, 10), minor=True); ax.set_yticks(np.arange(-0.5, 10), minor=True)
ax.grid(which="minor", color="#d5dae1", lw=0.6); ax.tick_params(which="both", length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.add_patch(Rectangle((-0.5, -0.5), 4, 4, fc="none", ec=BRICK, lw=2.0, zorder=5))
ax.add_patch(Rectangle((6.5, 6.5), 3, 3, fc="none", ec=TEAL, lw=2.0, zorder=5))
ax.annotate("윗옷 네 클래스\n서로 바꿔 부른다", xy=(3.5, 1.5), xytext=(8.0, 1.5), fontsize=9, color=BRICK,
            ha="center", va="center", weight="bold", linespacing=1.5,
            arrowprops=dict(arrowstyle="->", color=BRICK, lw=1.0))
ax.annotate("신발 세 클래스\n가장 큰 칸이 5%다", xy=(6.5, 8.0), xytext=(2.6, 8.0), fontsize=9,
            color=TEAL, ha="center", va="center", weight="bold", linespacing=1.5,
            arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.0))
fig.tight_layout()
fig.savefig("fig_9_10.png", dpi=300)
plt.show()
print([(names[r], names[c], round(float(share[r, c]))) for r in range(10) for c in range(10)
       if r != c and round(share[r, c]) >= 5])

In [ ]:
# 그림 9-11
first_net = nets["합성곱 신경망"][0]
first_net.eval()
with torch.no_grad():
    chance = torch.cat([torch.softmax(first_net(X_va[k:k + 1000]), dim=1) for k in range(0, 10000, 1000)])
said, truth = chance.argmax(dim=1).numpy(), y_va.numpy()
hit_t, miss, hit_s = (truth == 0) & (said == 0), (truth == 6) & (said == 0), (truth == 6) & (said == 6)
n_t, n_s = int((truth == 0).sum()), int((truth == 6).sum())
groups = [(f"정답도 예측도 티셔츠 (티셔츠 {n_t:,}장 가운데 {int(hit_t.sum())}장)", hit_t, NAVY),
          (f"정답은 셔츠, 예측은 티셔츠 (셔츠 {n_s:,}장 가운데 {int(miss.sum())}장)", miss, BRICK),
          (f"정답도 예측도 셔츠 (셔츠 {n_s:,}장 가운데 {int(hit_s.sum())}장)", hit_s, TEAL)]
fig, axes = plt.subplots(3, 7, figsize=(7, 5.4))
fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.12, wspace=0.08, hspace=0.62)
for r, (word, mask, col) in enumerate(groups):
    for c, row in enumerate(np.flatnonzero(mask)[:7]):
        ax = axes[r, c]
        ax.imshow(images[valid_rows[row]], cmap="gray_r", vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        for side in ax.spines.values():
            side.set_color(col); side.set_linewidth(1.0)
        ax.set_xlabel(f"티셔츠 {chance[row, 0] * 100:.0f}%\n셔츠 {chance[row, 6] * 100:.0f}%", fontsize=9, labelpad=2,
                      color=col)
    axes[r, 0].set_title(word, fontsize=9, color=col, loc="left", weight="bold")
fig.text(0.5, 0.015, "이미지 아래의 숫자는 이 모델이 티셔츠와 셔츠에 준 확률이다.", fontsize=9, color="#3a4350",
         ha="center")
fig.savefig("fig_9_11.png", dpi=300)
plt.show()
print(int(hit_t.sum()), n_t, int(miss.sum()), n_s, int(hit_s.sum()))
print([round(float(chance[row, 0]) * 100) for row in np.flatnonzero(miss)[:7]])